In [26]:
import pandas as pd
import altair as alt

df = pd.read_csv("https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv")

df.head()

,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


In [27]:
clean["Usage Description"].value_counts().head(8)

Usage Description
Storage               2095
Unusual               1615
Industrial            1142
Assembly               917
Detention & Correc     726
Business               694
Residential            628
Health Care            380
Name: count, dtype: int64

In [33]:
clean = df.dropna(subset=["Square Footage", "Year Constructed", "Usage Description"])

clean = clean[(clean["Square Footage"] > 0) & (clean["Year Constructed"].between(1800, 2026))].copy()

top_usages = clean["Usage Description"].value_counts().head(8).index.tolist()


In [34]:
plot1_data = clean[clean["Usage Description"].isin(top_usages)].groupby("Usage Description", as_index=False)["Square Footage"].median()

chart1 = alt.Chart(plot1_data).mark_bar().encode(x=alt.X("Square Footage:Q", title="Median Square Footage"), y=alt.Y("Usage Description:N", sort="-x", title="Building Usage"), 
                                                 color=alt.value("steelblue"), tooltip=[alt.Tooltip("Usage Description:N"), alt.Tooltip("Square Footage:Q", format=",.0f")]
                                                 ).properties(title="Median Building Size by Usage", width=550, height=300)

chart1

alt.Chart(...)

In [30]:
2019 // 10 * 10

2010

In [35]:
plot2_data = clean[(clean["Year Constructed"] >= 1900) & (clean["Usage Description"].isin(top_usages))].copy()

plot2_data["Decade"] = (plot2_data["Year Constructed"] // 10 * 10).astype(int)

decade_counts = plot2_data.groupby(["Decade", "Usage Description"]).size().reset_index(name="Count")

dropdown = alt.binding_select(options=top_usages, name="Building Usage: ")

selection = alt.selection_point(fields=["Usage Description"], bind=dropdown, value=[{"Usage Description":top_usages[0]}])

chart2 = alt.Chart(decade_counts).mark_bar().encode(x=alt.X("Decade:O", title="Decade Constructed"), y=alt.Y("Count:Q", title="Number of Buildings"),
                                                    color=alt.value("darkorange"), tooltip=["Decade:O", "Count:Q"]
                                                    ).add_params(selection).transform_filter(selection).properties(title="Buildings Contructed by Decade", width=550, height=300)

chart2

alt.Chart(...)

In [36]:
from pathlib import Path

repo = Path.cwd()

if repo.name == "python_notebooks":
    repo = repo.parent

output = repo / "assets" / "json"
output.mkdir(parents=True, exist_ok=True)

chart1.save(output / "building_sizes.json")
chart2.save(output / "building_decades.json")